# 65 · Gene models: Gaussian on log1p against negative binomial on counts

**The question.** The PT pathway screens rank genes by how much their species difference changes
along the proximal tubule (PT). Until now the gene model was a Gaussian regression spline on
log1p(count / library × 10⁴). From now on it is a negative-binomial (NB) GLM on raw counts. This
notebook is the tracked record of why.

**The problem with log1p.** On the log1p scale a constant species ratio is not a constant
difference. Where counts are low, or where a gene is nearly absent in one species, the log1p
curve bends. A Gaussian `T_spatial` can then score a pure species offset as a change of shape.
An NB model with a log link keeps a constant ratio constant.

**The data:**
- 12,272 PT cross-sections from 2 control mice and 2 cortex sections of 1 human donor, all male;
- 11,071 genes measured in both species;
- 1,513 Reactome, Hallmark and KEGG pathways;
- the SCF13 coordinate (notebook 13's scFates coordinate, as used by notebooks 37 and 61).

**What each section does:**
1. **Setup and inputs.**
2. **The Gaussian reference.** Recomputes the Gaussian gene statistics and both joint-test screens
   from code, and checks them against the values the other notebooks recorded.
3. **NB statistics.** Fits the NB models on counts and runs the same two screens.
4. **The method table.** Calls, relabeled calls, calibration and overlap, side by side.
5. **Gene-level diagnostics.** Which genes each model ranks high, and why. These checks use no
   model, so they are the main evidence.
6. **Simulation.** Pure species offsets, and a positional positive control.
7. **Dispersion shrinkage.** Does shrinking the NB dispersions change the calls?
8. **Supplementary figure.**
9. **Summary and run record.**

**Status.** The switch was decided on 2026-10-09, after a scratch pilot on these same data. It
is therefore post hoc; `docs/results/pt-gene-model-nb.md` freezes the NB specification and
records the deviation. This notebook recomputes the Gaussian side itself, so it still runs after
notebooks 37, 45 and 50 are re-run on NB.

**Assumptions.** SCF13 is treated as an exact coordinate. Specimens, not structures, are the
replicates. Gene statistics are ranks: their degrees of freedom count structures, so they are not
calibrated gene tests. All results are descriptive within this cohort.

## 1 · Setup and inputs

In [ ]:
import argparse
import glob
import json
import os
import sys
import time
from pathlib import Path

NOTEBOOK_START = time.perf_counter()
start = Path(__file__).resolve().parent if '__file__' in globals() else Path.cwd()
project = next(p for p in (start, *start.parents) if (p / 'pseudospace').is_dir())
sys.path.insert(0, str(project))
parser = argparse.ArgumentParser(add_help=False)
parser.add_argument('--data-root', type=Path)
parser.add_argument('--results-root', type=Path)
args, _ = parser.parse_known_args()
data_root = Path(args.data_root or os.environ.get('PSEUDOSPACE_DATA_ROOT') or project / 'data').expanduser()
results_root = Path(args.results_root or os.environ.get('PSEUDOSPACE_RESULTS_ROOT') or project / 'results').expanduser()
coordinate_file = results_root / 'pt_reconstruction_v2' / 'primary_pt_coordinate.csv'
ortholog_file = results_root / 'minimal_pt_scfates' / 'ortholog_map_used.csv'
nb37 = results_root / 'pt_pathway_final' / 'scfates'     # read only for first-run guards
nb45 = results_root / 'pt_revision_method'                # read only for first-run guards
nb50 = results_root / 'pt_primary_pathways' / 'tables'    # read only for first-run guards
output = results_root / 'pt_gene_model_comparison'
tables, figures, reference = output / 'tables', output / 'figures', output / 'reference'
for folder in (tables, figures, reference):
    folder.mkdir(parents=True, exist_ok=True)
LIBRARIES = ('Reactome_2022', 'MSigDB_Hallmark_2020', 'KEGG_2019_Mouse')
REQUIRED = [coordinate_file, ortholog_file, results_root / 'minimal_pt_scfates' / 'cross_species_pt_scfates.h5ad',
            *[data_root / 'mouse_vs_human' / 'pathway_gene_sets' / f'{lib}.json' for lib in LIBRARIES]]
for path in REQUIRED:
    if not path.is_file():
        raise FileNotFoundError(f'Missing input: {path} (run notebooks 13 and 36 first)')

NOTEBOOK_LOGIC_VERSION = '65.gene_model.1'   # Bump when a cached calculation changes.
ALPHA, NCDR_MAX, NULL_RATE = .05, .25, .05      # discovery level and notebook 31's relabeling rule
CALIBRATION_MAX = 1.5 * ALPHA                    # notebook 61's criterion 2: share of relabeled p <= 0.05
BASIS_DF, N_NULL, SEED = 6, 9999, 12            # notebook 37's spline and matched null
TOP_SHARE, ABSENT = .10, .01                     # top-10% gene rule; "nearly absent" = detected in < 1% of one species
PSEUDOBULK_BINS, MIN_STRUCTURES_PER_CELL = 8, 20  # notebook 37's split plot
N_SIM, SIM_AMPLITUDE, SIM_CENTRE, SIM_WIDTH = 1000, .5, .5, .15   # frozen simulation design
SIM_PICK_SEED, SIM_DRAW_SEED = 0, 1
JACCARD_MIN = .8                                 # shrinkage rule, fixed before the pilot check ran
N_JOBS = int(os.environ.get('PSEUDOSPACE_N_JOBS', '36'))   # parallel workers; never part of a cache key

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from joblib import Parallel, delayed, parallel_config
from scipy import sparse
from scipy.stats import norm, spearmanr
from statsmodels.stats.multitest import multipletests
from threadpoolctl import threadpool_limits

MM = 1 / 25.4
INK, MUTED, LIGHT, NULL = '#222222', '#666666', '#D9D9D9', '#9A9A9A'
GAUSS_COLOR, NB_COLOR, ABSENT_COLOR = '#eb6834', '#2a78d6', '#C0392B'   # one colour per gene model in every figure
plt.rcParams.update({
    'font.family': ['Liberation Sans', 'DejaVu Sans'], 'font.size': 6.5, 'axes.titlesize': 6.5,
    'axes.titleweight': 'bold', 'axes.labelsize': 6.5, 'xtick.labelsize': 6, 'ytick.labelsize': 6,
    'legend.fontsize': 6, 'legend.frameon': False, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.linewidth': .6, 'xtick.major.width': .6, 'ytick.major.width': .6, 'xtick.major.size': 2.5,
    'ytick.major.size': 2.5, 'lines.linewidth': 1.0, 'pdf.fonttype': 42, 'svg.fonttype': 'none', 'savefig.dpi': 300})
pd.set_option('display.width', 240, 'display.max_columns', 40, 'display.max_colwidth', 70, 'display.max_rows', 80)
print('Results folder:', output, '· parallel workers:', N_JOBS)

### 1.1 · Structures, genes, pathways and partitions

The construction is notebook 37's on SCF13 (`pseudospace.coordinate_inputs.pt_inputs`):
- the structures inside every specimen's 1st–99th percentile range of the coordinate;
- the genes measured in both species and detected in ≥ 2% of structures;
- the pathways with 10–300 tested genes.

The library size is the total over all orthologs measured in both species. The NB offset is its
log. The Gaussian outcome is log1p(count / library × 10⁴); a guard checks that identity.

A gene is **nearly absent** when it is detected in fewer than 1% of the structures of one
species.

In [ ]:
from pseudospace.coordinate_inputs import pt_inputs, read_coordinate
from pseudospace.pathway_calibration import balanced_partitions
from pseudospace.stage_cache import cached_frame, cached_payload, digest
from pseudospace.stats_gam import gam_internal_knots

t0 = time.perf_counter()
inputs = pt_inputs(results_root, data_root, read_coordinate(coordinate_file))
position, human, specimen, segment = (inputs[k] for k in ('position', 'human', 'specimen', 'segment'))
genes, gene_sets, strata, covariates = inputs['genes'], inputs['gene_sets'], inputs['strata'], inputs['covariates']
Y, counts, library, grid = inputs['Y'].tocsr(), inputs['counts'].tocsr(), inputs['library'], inputs['grid']
pathways = list(gene_sets)
SPECIMENS = {'mouse': ['Ctrl1A2', 'Ctrl1A4'], 'human': ['HUK1_COR1', 'HUK1_MED1']}
ALL_SPECIMENS = SPECIMENS['mouse'] + SPECIMENS['human']
partitions = balanced_partitions(specimen, np.where(human, 'human', 'mouse'))
SWAPS = [p for p in partitions if p != 'species']
PARTS = ['species', *SWAPS]
assert len(position) == 12272 and len(genes) == 11071 and len(pathways) == 1513, 'notebook 37 universe changed'
check = counts[:2000].multiply(1e4 / library[:2000, None]).tocsr()
check.data = np.log1p(check.data)
assert abs(check - Y[:2000]).max() < 1e-12, 'Y is not log1p(count / library x 1e4)'
OFFSET = np.log(library)
min_detection = covariates[['detection_mouse', 'detection_human']].min(axis=1).to_numpy()
NEARLY_ABSENT = min_detection < ABSENT
mean_expression = covariates.mean_expression.to_numpy()
detection = covariates.detection.to_numpy()

cache = output / 'stage_cache'
from pseudospace import nb_gam, pathway_calibration, pathway_remodeling, specimen_null, split_plot
code = digest([NOTEBOOK_LOGIC_VERSION] + [Path(m.__file__) for m in
                                           (nb_gam, pathway_calibration, pathway_remodeling, specimen_null, split_plot)])
input_key = {'counts': digest(counts), 'Y': digest(Y), 'position': position, 'specimen': specimen,
             'segment': segment, 'genes': genes}
SECONDS = {}   # uncached compute time of every cached stage


def cached(stage, compute, *, params=None, inputs=None):
    """cached_payload under this notebook's root, inputs and code; keeps the stage's uncached compute time."""
    def wrapped():
        t = time.perf_counter()
        out = dict(compute())
        out['__seconds'] = np.array(time.perf_counter() - t)
        return out
    result = cached_payload(stage, wrapped, root=cache, params=params, inputs={**input_key, **(inputs or {})}, code=code)
    SECONDS[stage] = float(result.pop('__seconds'))
    return result


def cached_table(stage, compute, *, params=None, inputs=None):
    """cached_frame counterpart of ``cached``."""
    def wrapped():
        t = time.perf_counter()
        return compute().assign(__seconds=time.perf_counter() - t)
    result = cached_frame(stage, wrapped, root=cache, params=params, inputs={**input_key, **(inputs or {})}, code=code)
    SECONDS[stage] = float(result['__seconds'].iloc[0])
    return result.drop(columns='__seconds')


def slug(p):
    return p.replace(':', '_').replace('+', '_')


def bh(p):
    return multipletests(np.nan_to_num(np.asarray(p, float), nan=1.), method='fdr_bh')[1]


def newest(folder, stage, kind='npz'):
    """Newest stage-cache entry of another notebook (first-run guards only)."""
    files = glob.glob(str(folder / f'{stage}__*.{kind}'))
    if not files:
        return None
    files.sort(key=lambda p: json.loads(Path(p + '.key.json').read_text())['written_utc'])
    if kind == 'csv':
        return pd.read_csv(files[-1])
    with np.load(files[-1]) as stored:
        return {k: stored[k] for k in stored.files}


def live_model(path):
    """'missing', 'nb.1' or 'gaussian' for a result table of another notebook (the NB re-runs write gene_model)."""
    if not Path(path).is_file():
        return 'missing'
    head = pd.read_csv(path, nrows=5)
    return 'nb.1' if 'gene_model' in head and head.gene_model.astype(str).eq('nb.1').any() else 'gaussian'


LIVE37 = live_model(nb37 / 'tables' / 'gene_level_statistics.csv')
LIVE_GAUSSIAN = LIVE37 == 'gaussian'
print(f'{len(position):,} structures · {len(genes):,} genes · {len(pathways):,} pathways · '
      f'{int(NEARLY_ABSENT.sum()):,} genes nearly absent in one species ({NEARLY_ABSENT.mean():.1%})')
print('Partitions:', partitions)
print(f"Notebook 37's live gene statistics: {LIVE37}. First-run guards against notebooks 37/45/50 "
      f"{'run' if LIVE_GAUSSIAN else 'are skipped'}. pt_inputs {time.perf_counter() - t0:.0f} s")

## 2 · The Gaussian reference

**Gene statistics.** For every gene and each of the three partitions (the species split and the
two relabelings), on log1p(count / library × 10⁴):
- `T_level`: partial F of a constant group offset;
- `T_spatial`: partial F of the group × position spline (6 df) beyond the offset;
- `T_steps`: partial F of group × S1/S2/S3 steps beyond the offset, computed on the structures.

Each group gets half the weight and each specimen an equal share. Specimen intercepts are
within-group contrasts. In a relabeling, the species' own shape enters as a nuisance term. This
is notebook 37's model (continuous) and notebook 45's step model (steps).

**The two pathway screens.** Both use the **joint test**: a covariate-matched rank test whose z
is divided by √VIF, the variance inflation from the correlation of the pathway's genes.
- **A6·JNT** ranks genes by `T_spatial`. The null is 9,999 random gene sets matched on
  expression, detection and coverage (seed 12). The VIF comes from the residuals of the species
  full model (notebook 37).
- **A4·JNT** ranks genes by `T_steps`. The null moments are exact. The pathway sets, strata and
  VIF are those of notebook 45's "our cohort" pool. Its VIF comes from residuals centred within
  each specimen × segment cell. Notebook 45's pool is rebuilt here from the same structures.

A call is a positive effect with BH q ≤ 0.05 within the partition.

**Guards.**
- **First run, while the other notebooks are still Gaussian:** every statistic and call must
  equal the value notebooks 37, 45 and 50 recorded. The check reads their result tables and
  stage caches, and is skipped once notebook 37's gene table carries `gene_model = nb.1`.
- **Every run:** the reference is saved in `reference/` on the first run. Later runs must
  reproduce it.

In [ ]:
from pseudospace.pathway_calibration import contrast_designs, nested_partial_f, normal_matched_q
from pseudospace.pathway_decoys import joint_matched_test
from pseudospace.pathway_remodeling import (correlation_adjusted_rank_tests, fit_nested_trajectories,
                                            matched_pathway_tests, matching_strata, residual_pathway_correlations)
from pseudospace.pathways import build_pathway_membership
from pseudospace.specimen_null import matched_auc_null, segment_designs, summary_partial_f

COMPARE = {'T_level': ('base', 'level'), 'T_spatial': ('level', 'full'), 'T_steps': ('seg_level', 'steps')}
knots = gam_internal_knots(position, basis_df=BASIS_DF)


def group_of(label):
    """(group, nuisance) per structure: notebook 37 cell 8."""
    if label == 'species':
        return human.astype(float), None
    return np.isin(specimen, partitions[label]).astype(float), human.astype(float)


def designs_of(label):
    """Continuous designs (notebook 37) and structure-level segment-step designs (notebook 45) for one partition."""
    group, nuisance = group_of(label)
    continuous, weights = contrast_designs(position, group, specimen, knots, nuisance_shape=nuisance)
    per_specimen = lambda v: {n: float(v[specimen == n][0]) for n in ALL_SPECIMENS}   # noqa: E731
    steps = segment_designs(specimen, segment, per_specimen(group), None if nuisance is None else per_specimen(nuisance))
    return {'base': continuous['base'], 'level': continuous['level'], 'full': continuous['full'],
            'seg_level': steps['level'], 'steps': steps['steps']}, weights


DESIGNS, W = {}, None
for p in PARTS:
    DESIGNS[p], w = designs_of(p)
    W = w if W is None else W
    assert np.allclose(W, w), 'the prior weights must be the same in every partition'
P_BASE = DESIGNS['species']['base'].shape[1] - 2            # intercept + spline columns (2 specimen contrasts follow)
assert P_BASE == 7 and DESIGNS['species']['full'].shape[1] == 16

GAUSS = {p: pd.DataFrame(cached(f'gauss_scores_{slug(p)}', lambda p=p: {
    k: v for k, v in nested_partial_f(Y, DESIGNS[p], W, COMPARE).items() if k != 'df'},
    params={'basis_df': BASIS_DF, 'label': p, 'compare': COMPARE}), index=genes) for p in PARTS}

# A6 VIF: residual correlations under the species full model (notebook 37 cell 10)
_held = {}


def gauss_species_fit():
    if 'fit' not in _held:
        _held['fit'] = fit_nested_trajectories(Y, position, human, specimen, grid, basis_df=BASIS_DF, return_residuals=True)
    return _held['fit']


gauss_fit = cached('gauss_species_fit', lambda: {**{k: gauss_species_fit()[k] for k in ('T_level', 'T_spatial', 'beta_human_level')},
                                                 'mean_residual': gauss_species_fit()['residuals'].mean(axis=1)},
                   params={'basis_df': BASIS_DF, 'grid': grid})
gauss_corr = cached_table('gauss_residual_correlations',
                          lambda: residual_pathway_correlations(gauss_species_fit()['residuals'], specimen, genes, gene_sets),
                          params={'basis_df': BASIS_DF}, inputs={'sets': gene_sets})
_held.clear()
np.testing.assert_allclose(gauss_fit['T_spatial'], GAUSS['species'].T_spatial, rtol=1e-6, atol=1e-8)
CAMERA_GAUSS = correlation_adjusted_rank_tests(GAUSS['species'].T_spatial, gene_sets, gauss_corr).set_index(
    'pathway_id').reindex(pathways)
VIF_GAUSS = CAMERA_GAUSS.variance_inflation

# A4 inputs: notebook 45's "our cohort" pool (cells 4 and 10), rebuilt from these structures
universe45 = pd.concat([build_pathway_membership(
    json.loads((data_root / 'mouse_vs_human' / 'pathway_gene_sets' / f'{lib}.json').read_text()), genes,
    ortholog_map=pd.read_csv(ortholog_file), library_name=lib, tested=genes, min_genes=1) for lib in LIBRARIES],
    ignore_index=True)
universe45['pathway_id'] = universe45.library + '::' + universe45.pathway
universe45 = universe45[universe45.n_tested.between(10, min(300, len(genes) - 1))]
SETS45 = {row.pathway_id: list(row.genes_present) for row in universe45.itertuples()}
PATHWAYS45 = list(SETS45)
assert sorted(PATHWAYS45) == sorted(pathways), 'notebook 45 tests the same 1,513 pathways'


def step_pool():
    """Equal-donor covariates (coverage = share of segments with any detection), strata, and the VIF from
    residuals centred within specimen x segment cells: notebook 45's build_pool for our cohort."""
    per_donor, corr = {}, []
    for name in ALL_SPECIMENS:
        rows = Y[specimen == name]
        seg = segment[specimen == name]
        present = np.array([np.asarray((rows[seg == s] > 0).sum(axis=0)).ravel() > 0 for s in np.unique(seg)])
        per_donor[name] = (np.asarray(rows.mean(axis=0)).ravel(), np.asarray((rows > 0).mean(axis=0)).ravel(),
                           present.mean(axis=0))
        dense = rows.toarray()
        for s in np.unique(seg):
            dense[seg == s] -= dense[seg == s].mean(axis=0)
        corr.append(residual_pathway_correlations(dense, np.repeat(name, len(dense)), genes, SETS45))
    balanced = [np.mean([np.mean([per_donor[d][k] for d in SPECIMENS[sp]], axis=0) for sp in ('mouse', 'human')], axis=0)
                for k in range(3)]
    pool_cov = pd.DataFrame({'mean_expression': balanced[0], 'detection': balanced[1], 'coverage': balanced[2]}, index=genes)
    assert pool_cov.detection.ge(.02).all(), 'every universe gene is tested in the pool'
    camera = correlation_adjusted_rank_tests(pool_cov.mean_expression, SETS45, pd.concat(corr, ignore_index=True))
    camera = camera.set_index('pathway_id').reindex(PATHWAYS45)
    return {'strata': matching_strata(pool_cov, bins=3), 'vif': camera.variance_inflation.to_numpy(float),
            'rho': camera.rho_residual.to_numpy(float), 'pathways': np.array(PATHWAYS45),
            **{k: pool_cov[k].to_numpy() for k in pool_cov}}


POOL = cached('step_pool', step_pool, params={'detection': .02}, inputs={'sets': SETS45})
VIF_POOL = pd.Series(POOL['vif'], index=PATHWAYS45)
print(f'Median VIF: A6 (species full-model residuals) {VIF_GAUSS.median():.2f}; '
      f'A4 (within-cell residuals, notebook 45) {VIF_POOL.median():.2f}')

### 2.1 · The Gaussian screens

The matched test of A6 draws its random sets from a fixed seed. Each partition is run in its own
worker; the draws do not depend on the number of workers.

In [ ]:
def _matched_one(frame, sets, strata_values):
    with threadpool_limits(1):
        return matched_pathway_tests(frame, sets, strata_values, n_null=N_NULL, seed=SEED)


def matched_a6(stage, frames):
    """Notebook 37's matched Monte Carlo test for every column of frames[partition], one worker per partition."""
    def compute():
        with parallel_config(backend='loky', inner_max_num_threads=1):
            parts = Parallel(n_jobs=len(PARTS))(delayed(_matched_one)(frames[p], gene_sets, strata) for p in PARTS)
        return pd.concat([t.assign(partition=p) for p, t in zip(PARTS, parts)], ignore_index=True)
    return cached_table(stage, compute, params={'n_null': N_NULL, 'seed': SEED},
                        inputs={'scores': {p: frames[p] for p in PARTS}, 'sets': gene_sets, 'strata': strata})


def a6_tests(matched, column, vif):
    """Joint test (z / sqrt(VIF), BH per partition) for one statistic of matched_a6's table."""
    t = joint_matched_test(normal_matched_q(matched[matched.statistic.eq(column)]), vif)
    return {p: t[t.partition.eq(p)].set_index('pathway_id').reindex(pathways) for p in PARTS}


def a4_tests(statistic, vif):
    """Notebook 61 cell 22: exact matched moments with notebook 45's sets and strata; z / sqrt(VIF), BH per partition."""
    out = {}
    for p in PARTS:
        t = matched_auc_null(pd.DataFrame({'T_steps': np.asarray(statistic[p], float)}, index=genes), SETS45, POOL['strata'])
        out[p] = joint_matched_test(t.assign(partition=p), vif).set_index('pathway_id').reindex(pathways)
    return out


def calls_of(tests):
    return {p: set(tests[p].index[tests[p].effect.gt(0) & tests[p].q_joint.le(ALPHA)]) for p in PARTS}


MATCHED_GAUSS = matched_a6('a6_matched_gaussian', {p: GAUSS[p][['T_spatial']] for p in PARTS})
TESTS, CALLS = {}, {}
TESTS['A6 Gaussian'] = a6_tests(MATCHED_GAUSS, 'T_spatial', VIF_GAUSS)
TESTS['A4 Gaussian'] = a4_tests({p: GAUSS[p].T_steps for p in PARTS}, VIF_POOL)
for key in ('A6 Gaussian', 'A4 Gaussian'):
    CALLS[key] = calls_of(TESTS[key])
    print(f'{key}·JNT calls (species, relabelings):', [len(CALLS[key][p]) for p in PARTS])
assert [len(CALLS['A6 Gaussian'][p]) for p in PARTS] == [60, 2, 0], 'Gaussian A6·JNT reference changed'
assert [len(CALLS['A4 Gaussian'][p]) for p in PARTS] == [50, 0, 0], 'Gaussian A4·JNT reference changed'

### 2.2 · Guards: the live notebooks (first run) and this notebook's own reference

In [ ]:
guard_log = []
if LIVE_GAUSSIAN:
    gene37 = pd.read_csv(nb37 / 'tables' / 'gene_level_statistics.csv', index_col=0).T_spatial.reindex(genes)
    np.testing.assert_allclose(GAUSS['species'].T_spatial, gene37, rtol=1e-6)
    guard_log.append('species T_spatial = notebook 37 gene_level_statistics.csv (rtol 1e-6)')
    for p in PARTS:
        stored = newest(nb37 / 'stage_cache', 'gene_scores_' + slug(p))
        if stored is not None:
            for column in ('T_level', 'T_spatial'):
                np.testing.assert_allclose(GAUSS[p][column], stored[column], rtol=1e-6, atol=1e-8)
    guard_log.append('T_level and T_spatial of all three partitions = notebook 37 gene_scores caches (rtol 1e-6)')
    table37 = pd.read_csv(nb37 / 'tables' / 'table_S2_pathways.csv').set_index('pathway_id')
    np.testing.assert_allclose(VIF_GAUSS, table37.vif.reindex(pathways), rtol=1e-6)
    guard_log.append('A6 VIF = notebook 37 Table S2 (rtol 1e-6)')
    tests37 = pd.read_csv(nb37 / 'tables' / 'matched_and_joint_tests_all_partitions.csv')
    for p in PARTS:
        ref = tests37[tests37.statistic.eq('T_spatial') & tests37.partition.eq(p)].set_index('pathway_id').reindex(pathways)
        np.testing.assert_allclose(TESTS['A6 Gaussian'][p].z_joint, ref.z_joint, rtol=1e-9)
        assert CALLS['A6 Gaussian'][p] == set(ref.index[ref.effect.gt(0) & ref.q_joint.le(ALPHA)])
    guard_log.append('A6·JNT joint z (rtol 1e-9) and calls [60, 2, 0] = notebook 37')
    pool45 = newest(nb45 / 'stage_cache', 'pool_our_cohort')
    if pool45 is not None and 'gene_model' not in pool45:
        assert list(pool45['genes'].astype(str)) == list(genes) and list(pool45['pathways'].astype(str)) == PATHWAYS45
        assert np.array_equal(np.asarray(pool45['strata']), POOL['strata'])
        np.testing.assert_allclose(POOL['vif'], pool45['vif'], rtol=1e-6)
        donors45 = list(pool45['donors'].astype(str))
        species_of = {d: float(d in SPECIMENS['human']) for d in donors45}
        for p in PARTS:
            group = species_of if p == 'species' else {d: float(d in partitions[p]) for d in donors45}
            summary = summary_partial_f(pool45, donors45, group, None if p == 'species' else species_of)
            np.testing.assert_allclose(GAUSS[p].T_steps, summary['T_steps'], rtol=1e-6, atol=1e-8)
        guard_log.append("notebook 45's pool: same pathways and strata, VIF (rtol 1e-6), and T_steps from its cell "
                         'summaries (rtol 1e-6) for all three partitions')
    if live_model(nb50 / 'cohort_step_screen_per_pathway.csv') == 'gaussian':
        steps50 = pd.read_csv(nb50 / 'cohort_step_screen_per_pathway.csv').set_index('pathway_id')
        assert CALLS['A4 Gaussian']['species'] == set(steps50.index[steps50.called.astype(bool)])
        guard_log.append('A4·JNT species calls = notebook 50 cohort step screen')
    if live_model(nb45 / 'tables' / 'anchor_our_cohort.csv') == 'gaussian':
        anchor = pd.read_csv(nb45 / 'tables' / 'anchor_our_cohort.csv').set_index('strategy').loc['P1 T_steps joint']
        assert [len(CALLS['A4 Gaussian'][p]) for p in PARTS] == [anchor.species, anchor.relabel_Ctrl1A2, anchor.relabel_Ctrl1A4]
        guard_log.append('A4·JNT calls [50, 0, 0] = notebook 45 anchor')
else:
    guard_log.append(f"live guards skipped: notebook 37's gene table is {LIVE37}")

# This notebook's own Gaussian reference: written once, reproduced on every later run.
ref_stats = pd.DataFrame({f'{c}|{p}': GAUSS[p][c] for p in PARTS for c in COMPARE}, index=pd.Index(genes, name='gene'))
ref_vif = pd.DataFrame({'vif_A6_species_full_residuals': VIF_GAUSS,
                        'vif_A4_within_cell_residuals': VIF_POOL.reindex(pathways)}).rename_axis('pathway_id')
ref_calls = pd.DataFrame([{'method': k, 'partition': p, 'pathway_id': x} for k in ('A6 Gaussian', 'A4 Gaussian')
                          for p in PARTS for x in sorted(CALLS[k][p])])
REF_FILES = {'stats': reference / 'gaussian_gene_statistics.csv', 'vif': reference / 'gaussian_vif.csv',
             'calls': reference / 'gaussian_calls.csv'}
HAVE_REFERENCE = all(ref_file.is_file() for ref_file in REF_FILES.values())
if HAVE_REFERENCE:
    saved = pd.read_csv(REF_FILES['stats'], index_col=0)
    assert list(saved.index) == genes and list(saved.columns) == list(ref_stats.columns)
    np.testing.assert_allclose(ref_stats.to_numpy(), saved.to_numpy(), rtol=1e-6, atol=1e-8)
    np.testing.assert_allclose(ref_vif.to_numpy(), pd.read_csv(REF_FILES['vif'], index_col=0).reindex(pathways).to_numpy(), rtol=1e-6)
    saved_calls = pd.read_csv(REF_FILES['calls'])
    assert set(map(tuple, saved_calls.to_numpy())) == set(map(tuple, ref_calls.to_numpy()))
    guard_log.append("this notebook's saved Gaussian reference reproduced (statistics and VIF rtol 1e-6, calls exact)")
elif LIVE_GAUSSIAN:
    ref_stats.to_csv(REF_FILES['stats'])
    ref_vif.to_csv(REF_FILES['vif'])
    ref_calls.to_csv(REF_FILES['calls'], index=False)
    guard_log.append('Gaussian reference written to reference/ (checked against the live notebooks above)')
else:
    raise RuntimeError('No saved Gaussian reference and the live notebooks are no longer Gaussian: nothing to check against.')
print('\n'.join('· ' + line for line in guard_log))

## 3 · NB statistics on raw counts

**The model** (`pseudospace/nb_gam.py`, `GENE_MODEL = 'nb.1'`):
- an NB GLM per gene on raw counts, with log link and offset log(library);
- the same designs and prior weights as section 2.

**Dispersion.** One α per gene, estimated by weighted maximum likelihood under the species full
model on all structures. There is no shrinkage. The same α is reused for every nested model and
every partition.

**Statistic.** The likelihood ratio, LR = max(D_reduced − D_full, 0), replaces the partial F. The
quasi-likelihood F, (LR / df₁) / (D_full / df₂), is computed and reported, and used for nothing.

**Failures.**
- A gene is **non-converged** when a fit fails R's glm criterion, its LR is not finite, or α does
  not converge. Its statistics are replaced by the median of the converged genes.
- A gene with no counts in one species is **separated**. Its limiting deviances are valid, so it
  is flagged and never filled.

**VIF.** For the NB screens the VIF comes from the Pearson residuals, (y − μ)/√(μ + αμ²), of the
species full model. This is the frozen rule, and it is applied here to the A6 sets and to the A4
sets. The Gaussian VIFs of section 2 are kept as a sensitivity. With the Gaussian VIF, the pilot
found 31 A6 calls and 15 A4 calls. This is the first run of the NB screens with the NB VIF.

**Two VIF diagnostics** follow the fits:
- the residual correlation and VIF of each residual type;
- the correlation, within each specimen, between a structure's mean residual over all genes and
  its log library size. A shared depth component makes every pair of genes look correlated.

In [ ]:
from pseudospace.nb_gam import GENE_MODEL, nb_dispersion, nb_fit, nb_species_trajectories, nested_nb_lr

DISPERSION = cached('nb_dispersion', lambda: nb_dispersion(counts, DESIGNS['species']['full'], W, OFFSET, n_jobs=N_JOBS),
                    params={'design': 'species M_full', 'basis_df': BASIS_DF})
assert str(DISPERSION['gene_model']) == GENE_MODEL
ALPHA_NB = {'alpha': DISPERSION['alpha'], 'converged': DISPERSION['converged'].astype(bool)}
NB = {}
for p in PARTS:
    scores = cached(f'nb_lr_{slug(p)}', lambda p=p: {k: v for k, v in nested_nb_lr(
        counts, DESIGNS[p], W, OFFSET, ALPHA_NB, COMPARE, quasi=True, n_jobs=N_JOBS).items() if k != 'df'},
        params={'basis_df': BASIS_DF, 'label': p, 'compare': COMPARE}, inputs={'alpha': ALPHA_NB})
    NB[p] = pd.DataFrame(scores, index=genes)
_held_nb = {}


def nb_species_fit():
    if 'fit' not in _held_nb:
        _held_nb['fit'] = nb_species_trajectories(counts, library, position, human, specimen, grid, ALPHA_NB,
                                                  basis_df=BASIS_DF, return_residuals=True, n_jobs=N_JOBS)
    return _held_nb['fit']


NB_FIT = cached('nb_species_fit', lambda: {**{k: nb_species_fit()[k] for k in
                                              ('T_level', 'T_spatial', 'beta_human_level', 'se_human_level', 'converged', 'separated')},
                                           'mean_residual': nb_species_fit()['residuals'].mean(axis=1)},
                params={'basis_df': BASIS_DF, 'grid': grid}, inputs={'alpha': ALPHA_NB})
nb_corr = cached_table('nb_pearson_residual_correlations', lambda: pd.concat(
    [residual_pathway_correlations(nb_species_fit()['residuals'], specimen, genes, sets).assign(family=family)
     for family, sets in (('A6', gene_sets), ('A4', SETS45))], ignore_index=True),
    params={'basis_df': BASIS_DF}, inputs={'alpha': ALPHA_NB, 'sets': [gene_sets, SETS45]})
_held_nb.clear()
CAMERA_NB = {family: correlation_adjusted_rank_tests(NB['species'][stat], sets, nb_corr[nb_corr.family.eq(family)].drop(
    columns='family')).set_index('pathway_id').reindex(list(sets))
    for family, stat, sets in (('A6', 'T_spatial', gene_sets), ('A4', 'T_steps', SETS45))}
VIF_NB, VIF_NB_A4 = CAMERA_NB['A6'].variance_inflation, CAMERA_NB['A4'].variance_inflation
LOG_RATIO = pd.Series(NB_FIT['beta_human_level'], index=genes)        # human / mouse, natural log, M_level
converged = NB['species'].converged.astype(bool).to_numpy()
separated = NB['species'].separated.astype(bool).to_numpy()
fit_check = np.abs(NB_FIT['T_spatial'] - NB['species'].T_spatial.to_numpy())
print(f"Dispersion: median α {np.median(DISPERSION['alpha']):.4f}; not converged {int((~ALPHA_NB['converged']).sum())}; "
      f"at the lower bound (no detectable overdispersion) {int(DISPERSION['at_lower'].sum())}")
display(pd.DataFrame({p: {'converged': int(NB[p].converged.sum()), 'non-converged (median-filled)': int((~NB[p].converged.astype(bool)).sum()),
                          'separated (flagged, not filled)': int(NB[p].separated.sum())} for p in PARTS}).T)
print(f'Species T_spatial from nb_species_trajectories against nested_nb_lr: max |difference| {fit_check.max():.2e} '
      f'(Spearman {spearmanr(NB_FIT["T_spatial"], NB["species"].T_spatial)[0]:.6f})')
assert spearmanr(NB_FIT['T_spatial'], NB['species'].T_spatial)[0] > .9999
VIF_SUMMARY = pd.DataFrame({name: {'median residual ρ': float(np.nanmedian(rho_values)), 'median VIF': float(vif.median()),
                                    '90th percentile VIF': float(vif.quantile(.9)), 'max VIF': float(vif.max())}
                             for name, rho_values, vif in (
    ('A6 sets · Gaussian residuals, species full model', CAMERA_GAUSS.rho_residual, VIF_GAUSS),
    ('A6 sets · NB Pearson residuals, species full model', CAMERA_NB['A6'].rho_residual, VIF_NB),
    ('A4 sets · Gaussian residuals within specimen × segment', POOL['rho'], VIF_POOL),
    ('A4 sets · NB Pearson residuals, species full model', CAMERA_NB['A4'].rho_residual, VIF_NB_A4))}).T
VIF_SUMMARY.to_csv(tables / 'vif_by_residual_type.csv')
display(VIF_SUMMARY.round(4))
print(f'Spearman of the A6 VIFs, Gaussian against NB Pearson residuals: {spearmanr(VIF_NB, VIF_GAUSS)[0]:.3f}')
DEPTH = pd.DataFrame({name: {s: float(np.corrcoef(fit['mean_residual'][specimen == s], OFFSET[specimen == s])[0, 1])
                             for s in ALL_SPECIMENS} for name, fit in (('Gaussian log1p residuals', gauss_fit),
                                                                       ('NB Pearson residuals', NB_FIT))})
DEPTH.to_csv(tables / 'mean_residual_vs_log_library.csv')
display(DEPTH.round(3).rename_axis('Pearson r of the mean residual over genes with log library, within specimen'))

### 3.1 · The NB screens

Three NB versions of each screen:
- **NB** (primary): the LR statistic with the NB Pearson-residual VIF;
- **NB, Gaussian VIF** (sensitivity): the same statistic with section 2's VIF, as in the pilot;
- **NB QL F** (reported only): the quasi-likelihood F with the NB VIF.

In [ ]:
MATCHED_NB = matched_a6('a6_matched_nb', {p: NB[p][['T_spatial', 'T_spatial_ql']] for p in PARTS})
TESTS['A6 NB'] = a6_tests(MATCHED_NB, 'T_spatial', VIF_NB)
TESTS['A6 NB, Gaussian VIF'] = a6_tests(MATCHED_NB, 'T_spatial', VIF_GAUSS)
TESTS['A6 NB QL F'] = a6_tests(MATCHED_NB, 'T_spatial_ql', VIF_NB)
TESTS['A4 NB'] = a4_tests({p: NB[p].T_steps for p in PARTS}, VIF_NB_A4)
TESTS['A4 NB, Gaussian VIF'] = a4_tests({p: NB[p].T_steps for p in PARTS}, VIF_POOL)
TESTS['A4 NB QL F'] = a4_tests({p: NB[p].T_steps_ql for p in PARTS}, VIF_NB_A4)
for key in TESTS:
    CALLS[key] = calls_of(TESTS[key])
PILOT = {'A6 Gaussian': [60, 2, 0], 'A4 Gaussian': [50, 0, 0], 'A6 NB, Gaussian VIF': [31, 0, 0], 'A4 NB, Gaussian VIF': [15, 1, 1]}
pilot_check = pd.DataFrame([{'method': k, 'this notebook': [len(CALLS[k][p]) for p in PARTS], 'pilot': v,
                             'same': [len(CALLS[k][p]) for p in PARTS] == v} for k, v in PILOT.items()]).set_index('method')
display(pilot_check)

## 4 · The method table

For each screen and gene model:
- **species calls** and **relabeled calls** (the two balanced specimen relabelings);
- **NCDR**: mean relabeled calls ÷ species calls;
- **relabeling rule** (notebook 31): NCDR < 0.25 and each relabeling below 5% of tested pathways;
- **calibration share** (notebook 61's criterion 2): the share of the 2 × 1,513 pooled relabeled
  p ≤ 0.05. Under a relabeling no species difference is left, so a calibrated test gives about
  0.05. The limit is 0.075;
- **overlap and Jaccard** with the Gaussian A6·JNT calls;
- **continuous adds over steps**: pathways the A6 screen calls and the A4 screen of the same
  version does not, and the reverse.

**What it can tell you:** whether each model keeps the screens specific and calibrated, and how
much the call list moves. **What it cannot tell you:** which list is right. The relabelings mix
the species, so a species-linked artefact (depth, probe panel, nearly absent genes) passes them.
Sections 5 and 6 address that.

In [ ]:
ORDER = ['A6 Gaussian', 'A6 NB', 'A6 NB, Gaussian VIF', 'A6 NB QL F', 'A4 Gaussian', 'A4 NB', 'A4 NB, Gaussian VIF', 'A4 NB QL F']
ROLE = {'A6 Gaussian': 'superseded', 'A6 NB': 'frozen primary', 'A6 NB, Gaussian VIF': 'sensitivity',
        'A6 NB QL F': 'reported only', 'A4 Gaussian': 'superseded', 'A4 NB': 'conventional reference',
        'A4 NB, Gaussian VIF': 'sensitivity', 'A4 NB QL F': 'reported only'}
G60 = CALLS['A6 Gaussian']['species']


def method_row(key):
    sp, relabeled = CALLS[key]['species'], [len(CALLS[key][s]) for s in SWAPS]
    ncdr = np.mean(relabeled) / len(sp) if sp else np.nan
    share = float(np.mean(np.concatenate([TESTS[key][s].p_joint.to_numpy(float) for s in SWAPS]) <= ALPHA))
    row = {'screen': key[:2] + '·JNT', 'version': key[3:], 'role': ROLE[key], 'species calls': len(sp),
           'relabeled calls': ', '.join(map(str, relabeled)), 'NCDR': ncdr,
           'relabeling rule': 'pass' if sp and ncdr < NCDR_MAX and max(relabeled) < NULL_RATE * len(pathways) else 'fail',
           'calibration share': share, 'calibration': 'pass' if share <= CALIBRATION_MAX else 'fail',
           'overlap with Gaussian A6·JNT': len(sp & G60), 'Jaccard with Gaussian A6·JNT': len(sp & G60) / len(sp | G60)}
    if key.startswith('A6') and 'A4' + key[2:] in CALLS:
        steps = CALLS['A4' + key[2:]]['species']
        row.update({'continuous adds over steps': len(sp - steps), 'steps add over continuous': len(steps - sp)})
    return row


METHOD_TABLE = pd.DataFrame([method_row(k) for k in ORDER], index=ORDER)
METHOD_TABLE.to_csv(tables / 'method_table.csv')
display(METHOD_TABLE.round(4))

### 4.1 · The pathways called by either model (A6·JNT)

Status compares the Gaussian A6·JNT calls with the primary NB calls. For each pathway:
- **share nearly absent:** the share of its genes detected in < 1% of one species' structures;
- **median |log ratio|:** the median over its genes of the NB human/mouse log rate ratio (natural
  log, offset model);
- **share Gaussian-only top 10%:** the share of its genes in the Gaussian top 10% of `T_spatial`
  but not in the NB top 10%.

In [ ]:
NAMES = pd.Series({p: p.split('::', 1)[1] for p in pathways}).str.replace(r' R-HSA-\d+$', '', regex=True)
top = lambda v: np.asarray(v, float) >= np.quantile(np.asarray(v, float), 1 - TOP_SHARE)   # noqa: E731  (notebook 61's rule)
TOP_G, TOP_N = top(GAUSS['species'].T_spatial), top(NB['species'].T_spatial)
member_index = {p: pd.Index(genes).get_indexer(gene_sets[p]) for p in pathways}
union = sorted(G60 | CALLS['A6 NB']['species'])
called = pd.DataFrame(index=pd.Index(union, name='pathway_id'))
called['name'] = NAMES.reindex(union)
called['n_genes'] = [len(gene_sets[p]) for p in union]
called['status'] = ['both' if p in G60 and p in CALLS['A6 NB']['species'] else 'Gaussian only' if p in G60 else 'NB only'
                    for p in union]
for key, label in (('A6 Gaussian', 'Gaussian'), ('A6 NB', 'NB'), ('A6 NB, Gaussian VIF', 'NB, Gaussian VIF'),
                   ('A4 Gaussian', 'A4 Gaussian'), ('A4 NB', 'A4 NB')):
    called[f'q {label}'] = TESTS[key]['species'].q_joint.reindex(union).to_numpy()
    called[f'called {label}'] = [p in CALLS[key]['species'] for p in union]
abs_ratio = LOG_RATIO.abs().to_numpy()
called['share nearly absent'] = [float(NEARLY_ABSENT[member_index[p]].mean()) for p in union]
called['median |log ratio|'] = [float(np.median(abs_ratio[member_index[p]])) for p in union]
called['share Gaussian-only top 10%'] = [float((TOP_G & ~TOP_N)[member_index[p]].mean()) for p in union]
called['median member mean expression'] = [float(np.median(mean_expression[member_index[p]])) for p in union]
called = called.sort_values(['status', 'q NB'])
called.to_csv(tables / 'pathways_called_by_either_model.csv')
by_status = called.groupby('status').agg(pathways=('name', 'size'), median_share_nearly_absent=('share nearly absent', 'median'),
                                         median_abs_log_ratio=('median |log ratio|', 'median'),
                                         median_share_gaussian_only_top10=('share Gaussian-only top 10%', 'median'))
all_paths = pd.DataFrame({'pathways': len(pathways),
                          'median_share_nearly_absent': np.median([NEARLY_ABSENT[member_index[p]].mean() for p in pathways]),
                          'median_abs_log_ratio': np.median([np.median(abs_ratio[member_index[p]]) for p in pathways]),
                          'median_share_gaussian_only_top10': np.median([(TOP_G & ~TOP_N)[member_index[p]].mean() for p in pathways])},
                         index=['all 1,513 pathways'])
BY_STATUS = pd.concat([by_status, all_paths])
BY_STATUS.to_csv(tables / 'pathways_called_by_status.csv')
display(BY_STATUS.round(3))
with pd.option_context('display.max_rows', 120):
    display(called[['name', 'status', 'q Gaussian', 'q NB', 'q NB, Gaussian VIF', 'called A4 NB', 'share nearly absent',
                    'median |log ratio|']].round(3))

## 5 · Gene-level diagnostics

These checks compare the two gene rankings directly. They use no model of the counts, so they
are the main evidence for the switch.

**5.1 Rank correlations.** If the Gaussian `T_spatial` absorbs offsets, it should follow the
species offset (`T_level`, |log ratio|) more closely than the NB statistic does.

**5.2 The top 10% by model.** The genes ranked in the top 10% by both models, by the Gaussian
model only, and by the NB model only. Offset leakage predicts that the Gaussian-only genes carry
large offsets and are often nearly absent in one species.

**5.3 The nearly absent genes.** A gene detected in < 1% of one species' structures has almost
no positional information in that species, so its species difference can hardly change shape.

**5.4 An independent reference: the split-plot genes.** Notebook 37's split-plot test works on
specimen × bin pseudobulk with voom weights and judges the species × position term against the
specimens. Its significant genes (BH ≤ 0.05) do not depend on the gene-model switch. The
question is whether they are enriched for nearly absent genes, as the Gaussian top 10% is.

In [ ]:
gT, nT = GAUSS['species'].T_spatial.to_numpy(), NB['species'].T_spatial.to_numpy()
rho = lambda a, b: float(spearmanr(a, b)[0])   # noqa: E731
SPEARMAN = pd.DataFrame([
    {'comparison': 'T_spatial: Gaussian vs NB', 'Spearman': rho(gT, nT)},
    {'comparison': 'T_level: Gaussian vs NB', 'Spearman': rho(GAUSS['species'].T_level, NB['species'].T_level)},
    {'comparison': 'T_steps: Gaussian vs NB', 'Spearman': rho(GAUSS['species'].T_steps, NB['species'].T_steps)},
    *[{'comparison': f'{model} T_spatial vs {what}', 'Spearman': rho(stat, other)}
      for model, stat, level in (('Gaussian', gT, GAUSS['species'].T_level), ('NB', nT, NB['species'].T_level))
      for what, other in (('mean expression', mean_expression), ('detection', detection),
                          ('T_level (same model)', level), ('|NB log ratio|', abs_ratio))]]).set_index('comparison')
SPEARMAN.to_csv(tables / 'gene_spearman.csv')
display(SPEARMAN.round(3))

groups = {'both': TOP_G & TOP_N, 'Gaussian only': TOP_G & ~TOP_N, 'NB only': TOP_N & ~TOP_G,
          'Gaussian top 10%': TOP_G, 'NB top 10%': TOP_N, 'all genes': np.ones(len(genes), bool)}
COMPOSITION = pd.DataFrame({name: {'genes': int(m.sum()), 'median |log ratio|': float(np.median(abs_ratio[m])),
                                   'median mean expression': float(np.median(mean_expression[m])),
                                   'median lower species detection': float(np.median(min_detection[m])),
                                   'nearly absent genes': int(NEARLY_ABSENT[m].sum()),
                                   'share nearly absent': float(NEARLY_ABSENT[m].mean())} for name, m in groups.items()}).T
COMPOSITION.to_csv(tables / 'top10_composition.csv')
display(COMPOSITION.round(3))

# Split-plot genes (notebook 37 cell 42, species split only)
from pseudospace.specimen import pseudobulk_profiles
from pseudospace.split_plot import log_cpm, moderated_f, voom_weights
from pseudospace.stats_gam import make_gam_design


def split_plot_species():
    lo, hi = grid[0], grid[-1]
    inside = (position >= lo) & (position <= hi)
    scaled = np.clip((position - lo) / (hi - lo), 0, 1)
    stacked = sparse.hstack([counts, sparse.csr_matrix(library[:, None])], format='csr')
    pb = pseudobulk_profiles(stacked[inside], specimen[inside], scaled[inside], n_bins=PSEUDOBULK_BINS,
                             gene_names=genes + ['__library__'], min_structures_per_bin=MIN_STRUCTURES_PER_CELL)
    pb_counts = pb[['count_' + g for g in genes]].to_numpy(float)
    pb_library = pb['count___library__'].to_numpy(float)
    pb_specimen = pb.specimen.astype(str).to_numpy()
    basis = make_gam_design(pb.pseudospace_bin_centre.to_numpy(float), np.array([]))[:, 1:]
    y = log_cpm(pb_counts, pb_library)
    specimen_columns = (pb_specimen[:, None] == np.array(ALL_SPECIMENS)[None, :]).astype(float)
    pb_human = np.isin(pb_specimen, SPECIMENS['human']).astype(float)
    design = np.column_stack([specimen_columns, basis, pb_human[:, None] * basis])
    test = list(range(design.shape[1] - basis.shape[1], design.shape[1]))
    result = moderated_f(y, voom_weights(y, pb_library, design), design, test)
    return {'F': result['F'], 'p': result['p'], 'n_cells': np.array(len(pb))}


SPLIT = cached('split_plot_species', split_plot_species, params={'bins': PSEUDOBULK_BINS, 'min': MIN_STRUCTURES_PER_CELL})
stored_split = newest(nb37 / 'stage_cache', 'split_plot_species')
if stored_split is not None:   # the split plot does not depend on the gene-model switch, so this guard always applies
    np.testing.assert_allclose(SPLIT['p'], stored_split['p'], rtol=1e-6, atol=1e-12)
    print("Split-plot p-values = notebook 37's split_plot_species cache (rtol 1e-6)")
SPLIT_GENES = bh(SPLIT['p']) <= ALPHA
ABSENT_ROWS = pd.DataFrame([
    {'gene list': 'all genes (base rate)', 'genes': len(genes), 'nearly absent': int(NEARLY_ABSENT.sum())},
    {'gene list': 'Gaussian top 10% T_spatial', 'genes': int(TOP_G.sum()), 'nearly absent': int((NEARLY_ABSENT & TOP_G).sum())},
    {'gene list': 'NB top 10% T_spatial', 'genes': int(TOP_N.sum()), 'nearly absent': int((NEARLY_ABSENT & TOP_N).sum())},
    {'gene list': 'split-plot significant (BH ≤ 0.05)', 'genes': int(SPLIT_GENES.sum()),
     'nearly absent': int((NEARLY_ABSENT & SPLIT_GENES).sum())}]).set_index('gene list')
ABSENT_ROWS['share nearly absent'] = ABSENT_ROWS['nearly absent'] / ABSENT_ROWS.genes
ABSENT_ROWS.to_csv(tables / 'nearly_absent_by_gene_list.csv')
display(ABSENT_ROWS.round(3))
print(f'Of the {int(NEARLY_ABSENT.sum())} nearly absent genes, {int((NEARLY_ABSENT & (nT < 1)).sum())} have NB T_spatial LR < 1 '
      f'and {int((NEARLY_ABSENT & separated).sum())} are separated (no counts at all in one species).')

GENE_TABLE = pd.DataFrame({'T_level_gauss': GAUSS['species'].T_level, 'T_spatial_gauss': gT, 'T_steps_gauss': GAUSS['species'].T_steps,
                           'T_level_nb': NB['species'].T_level, 'T_spatial_nb': nT, 'T_steps_nb': NB['species'].T_steps,
                           'T_spatial_nb_ql': NB['species'].T_spatial_ql, 'nb_log_ratio_human_mouse': LOG_RATIO,
                           'dispersion': DISPERSION['alpha'], 'dispersion_converged': ALPHA_NB['converged'],
                           'nb_converged': converged, 'nb_separated': separated, 'mean_expression': mean_expression,
                           'detection': detection, 'detection_mouse': covariates.detection_mouse.to_numpy(),
                           'detection_human': covariates.detection_human.to_numpy(), 'nearly_absent': NEARLY_ABSENT,
                           'top10_gauss': TOP_G, 'top10_nb': TOP_N, 'split_plot_q': bh(SPLIT['p'])},
                          index=pd.Index(genes, name='gene'))
for p in SWAPS:
    for c in ('T_spatial', 'T_steps'):
        GENE_TABLE[f'{c}_gauss|{p}'] = GAUSS[p][c].to_numpy()
        GENE_TABLE[f'{c}_nb|{p}'] = NB[p][c].to_numpy()
GENE_TABLE.to_csv(tables / 'gene_statistics_gaussian_and_nb.csv')

## 6 · Simulation: pure species offsets and a positional positive control

**Design (frozen before the pilot ran).**
- 1,000 genes drawn at random from the 11,071 (`default_rng(0)`, without replacement).
- For each gene the species-split NB fit gives the mean:
  - **mouse structures:** the fitted full-model mouse curve of the structure's own specimen;
  - **human structures:** the mouse population curve (specimen contrasts at 0) plus the gene's
    constant human/mouse log ratio from the offset model.
- **Pure offset:** the two species differ only by that constant ratio. Any `T_spatial` signal is
  a false shape.
- **Positive control:** the human log mean also gets a bump, 0.5 × exp(−(s − 0.5)² / (2 · 0.15²)).
- Counts are NB with the gene's fitted α (gamma–Poisson, `default_rng(1)`, a fresh generator per
  scenario). Library sizes and positions are the real ones.
- Each simulated gene is scored by both models exactly as the real genes: the Gaussian partial F
  on log1p, and the NB LR after re-estimating α.
- A statistic counts as high when it reaches the real data's top-10% cut for that model.

**What it can tell you:** how often a pure offset reaches the top 10% under each model, and
whether the NB statistic still detects a real positional change.

**What it cannot tell you:** which model is right for the real data. The counts are drawn from
the NB model, so the simulation favours NB by construction. Section 5 is the model-free
evidence.

In [ ]:
PICK = np.sort(np.random.default_rng(SIM_PICK_SEED).choice(len(genes), N_SIM, replace=False))
X_SIM = {k: DESIGNS['species'][k] for k in ('level', 'full')}
THRESHOLD = {'gauss': float(np.quantile(gT, 1 - TOP_SHARE)), 'nb': float(np.quantile(nT, 1 - TOP_SHARE)),
             'ql': float(np.quantile(NB['species'].T_spatial_ql, 1 - TOP_SHARE))}


def nb_draw(log_mu, alpha, seed):
    """Gamma-Poisson draws with mean exp(log_mu) and dispersion alpha (one generator, as the pilot)."""
    rng = np.random.default_rng(seed)
    mu = np.exp(log_mu)
    shape = 1. / alpha[None, :]
    lam = rng.gamma(np.broadcast_to(shape, mu.shape), mu / shape)
    return rng.poisson(lam).astype(float)


def simulate():
    alpha = DISPERSION['alpha'][PICK]
    beta_full = nb_fit(counts[:, PICK], X_SIM['full'], W, OFFSET, alpha, n_jobs=N_JOBS, block_size=50)['beta']
    ratio = NB_FIT['beta_human_level'][PICK]
    specimen_curve = X_SIM['full'] @ beta_full.T                          # mouse rows: own specimen's mouse curve
    population = X_SIM['full'][:, :P_BASE] @ beta_full[:, :P_BASE].T      # intercept + spline, contrasts at 0
    log_mean = np.where(human[:, None], population + ratio[None, :], specimen_curve) + OFFSET[:, None]
    bump = SIM_AMPLITUDE * np.exp(-(position - SIM_CENTRE) ** 2 / (2 * SIM_WIDTH ** 2))
    out = {'ratio': ratio}
    for scenario, lm in (('offset', log_mean), ('control', log_mean + np.where(human, bump, 0.)[:, None])):
        y = nb_draw(lm, alpha, SIM_DRAW_SEED)
        out[f'{scenario}|gauss'] = nested_partial_f(np.log1p(y * (1e4 / library[:, None])), X_SIM, W,
                                                    {'T_spatial': ('level', 'full')})['T_spatial']
        disp = nb_dispersion(y, X_SIM['full'], W, OFFSET, n_jobs=N_JOBS, block_size=50)
        lr = nested_nb_lr(y, X_SIM, W, OFFSET, disp, {'T_spatial': ('level', 'full')}, quasi=True, fill=None,
                          n_jobs=N_JOBS, block_size=50)
        out.update({f'{scenario}|nb': lr['T_spatial'], f'{scenario}|ql': lr['T_spatial_ql'],
                    f'{scenario}|ok': lr['converged'], f'{scenario}|alpha_refit': disp['alpha'],
                    f'{scenario}|detection': (y > 0).mean(axis=0)})
    return out


SIM = cached('simulation', simulate, params={'n': N_SIM, 'seeds': (SIM_PICK_SEED, SIM_DRAW_SEED),
                                            'bump': (SIM_AMPLITUDE, SIM_CENTRE, SIM_WIDTH)}, inputs={'alpha': ALPHA_NB})
SCENARIO_NAME = {'offset': 'pure offset', 'control': 'positive control (human bump)'}
sim_genes, sim_rows = [], []
for scenario, title in SCENARIO_NAME.items():
    frame = pd.DataFrame({'gene': np.asarray(genes)[PICK], 'scenario': title, 'mean_expression': mean_expression[PICK],
                          'abs_log_ratio': np.abs(SIM['ratio']), 'T_spatial_gauss': SIM[f'{scenario}|gauss'],
                          'T_spatial_nb': SIM[f'{scenario}|nb'], 'T_spatial_nb_ql': SIM[f'{scenario}|ql'],
                          'nb_converged': SIM[f'{scenario}|ok'].astype(bool), 'dispersion_refit': SIM[f'{scenario}|alpha_refit']})
    frame['expression tertile'] = pd.qcut(frame.mean_expression, 3, labels=['low', 'mid', 'high']).astype(str)
    frame['offset tertile'] = pd.qcut(frame.abs_log_ratio, 3, labels=['small', 'mid', 'large']).astype(str)
    above = {m: frame[f'T_spatial_{c}'].ge(THRESHOLD[m]).to_numpy() for m, c in (('gauss', 'gauss'), ('nb', 'nb'), ('ql', 'nb_ql'))}
    frame['above_gauss'], frame['above_nb'] = above['gauss'], above['nb']
    sim_genes.append(frame)
    subsets = [('all', np.ones(N_SIM, bool))]
    subsets += [(f'expression {t}', frame['expression tertile'].eq(t).to_numpy()) for t in ('low', 'mid', 'high')]
    subsets += [(f'offset {t}', frame['offset tertile'].eq(t).to_numpy()) for t in ('small', 'mid', 'large')]
    for label, m in subsets:
        sim_rows.append({'scenario': title, 'genes': label, 'n': int(m.sum()),
                         'Gaussian share above cut': above['gauss'][m].mean(), 'NB share above cut': above['nb'][m].mean(),
                         'NB QL share above cut (reported only)': above['ql'][m].mean(),
                         'median Gaussian F': float(frame.T_spatial_gauss[m].median()),
                         'median NB LR': float(frame.T_spatial_nb[m].median()), 'NB non-converged': int((~frame.nb_converged[m]).sum())})
SIM_TABLE = pd.DataFrame(sim_rows)
SIM_TABLE.to_csv(tables / 'simulation_summary.csv', index=False)
pd.concat(sim_genes, ignore_index=True).to_csv(tables / 'simulation_genes.csv', index=False)
print('Real-data top-10% cuts:', {k: round(v, 2) for k, v in THRESHOLD.items()})
display(SIM_TABLE.round(3))

## 7 · Dispersion shrinkage: a sensitivity check

The primary α is each gene's own maximum-likelihood estimate. A DESeq2-style alternative shrinks
it toward a trend over mean expression. The rule was fixed before the pilot check ran: if the
unshrunk and the shrunk screens both pass the relabeling rule and the calibration limit, and their
species calls have Jaccard ≥ 0.8, the unshrunk estimate stays primary.

**Shrinkage (as the pilot's `shrink.py`):**
- **mean:** the prior-weighted mean of size-factor-normalised counts;
- **trend:** α_trend = a₀ + a₁ / mean, fitted as DESeq2's `parametricDispersionFit` (a Gamma GLM
  with identity link on the gene-wise α, refitted on genes with α / α_trend in (10⁻⁴, 15)). Genes
  at either α bound or with unsettled α are left out of the fit;
- **prior:** normal on log α around log α_trend. Its variance is max(MAD² − ψ₁((m − p)/2), 0.25)
  of the log residuals, with m the effective sample size of the prior weights and p = 16;
- **MAP:** IRLS for the coefficients and Newton for α alternate until α settles, from the ML α.

The shrunk α then replaces the unshrunk α in A6·JNT for all three partitions. The rule is
applied twice: with the frozen NB Pearson-residual VIF, and with the Gaussian VIF that the pilot
check used.

In [ ]:
import warnings

import statsmodels.api as sm
from scipy.special import digamma, polygamma

A_LO, A_HI = 1.0001e-8, 9999.
sf = library / np.exp(np.mean(np.log(library)))
base_mean = np.asarray(counts.T @ (W / sf)).ravel() / W.sum()
alpha_ml = DISPERSION['alpha']
use = (alpha_ml > A_LO) & (alpha_ml < A_HI) & ALPHA_NB['converged']
coefs = np.array([.1, 1.])
with warnings.catch_warnings():
    warnings.simplefilter('ignore')     # statsmodels warns that the identity link is not canonical for Gamma
    for iteration in range(11):
        ratio = alpha_ml / (coefs[0] + coefs[1] / base_mean)
        good = use & (ratio > 1e-4) & (ratio < 15)
        glm = sm.GLM(alpha_ml[good], np.column_stack([np.ones(good.sum()), 1. / base_mean[good]]),
                     family=sm.families.Gamma(link=sm.families.links.Identity())).fit(start_params=coefs)
        old, coefs = coefs, glm.params
        assert (coefs > 0).all(), f'non-positive trend coefficients {coefs}'
        if np.sum(np.log(coefs / old) ** 2) < 1e-6 and glm.converged:
            break
    else:
        raise RuntimeError('the dispersion trend did not converge')
TREND = coefs[0] + coefs[1] / base_mean
residual = np.log(alpha_ml[use]) - np.log(TREND[use])
mad2 = (1.4826 * np.median(np.abs(residual - np.median(residual)))) ** 2
m_eff = W.sum() ** 2 / (W ** 2).sum()
p_full = np.linalg.matrix_rank(DESIGNS['species']['full'])
SIGMA2 = max(mad2 - float(polygamma(1, (m_eff - p_full) / 2)), .25)
print(f'Trend α = {coefs[0]:.5f} + {coefs[1]:.5f} / mean ({iteration + 1} iterations, {int(good.sum())} of {int(use.sum())} '
      f'genes in the window); log-α residual MAD² {mad2:.4f}; m_eff {m_eff:.0f}; prior variance {SIGMA2:.4f}')


def map_alpha(y, mu, w, alpha0, log_trend, sigma2, *, maxit=60, tol=1e-7, gtol=1e-6):
    """Per-gene MAP of the NB dispersion given mu: weighted log-likelihood + log N(log alpha; log_trend, sigma2).
    nb_gam's safeguarded Newton on phi = log(1 / alpha), with the prior's gradient and curvature added."""
    phi = -np.log(np.clip(alpha0, nb_gam.A_MIN, nb_gam.A_MAX))
    lo, hi = -np.log(nb_gam.A_MAX), -np.log(nb_gam.A_MIN)
    done_all = np.zeros(len(phi), bool)
    active = np.arange(len(phi))

    def objective(ya, ma, ph, m):
        return nb_gam._loglik(ya, ma, np.exp(-ph), w) - (ph + m) ** 2 / (2 * sigma2)

    for _ in range(maxit):
        ya, ma, ph, m = y[:, active], mu[:, active], phi[active], log_trend[active]
        th = np.exp(ph)[None, :]
        tm = th + ma
        score = (w[:, None] * (digamma(ya + th) - digamma(th) + np.log(th) + 1. - np.log(tm) - (ya + th) / tm)).sum(0)
        hess = (w[:, None] * (polygamma(1, ya + th) - polygamma(1, th) + 1. / th - 2. / tm + (ya + th) / tm ** 2)).sum(0)
        t = th[0]
        g1 = t * score - (ph + m) / sigma2
        g2 = t * t * hess + t * score - 1. / sigma2
        step = np.clip(np.where(g2 < 0, -g1 / np.where(g2 < 0, g2, -1.), np.sign(g1)), -2., 2.)
        new = np.clip(ph + step, lo, hi)
        old_obj, new_obj = objective(ya, ma, ph, m), objective(ya, ma, new, m)
        for _ in range(20):
            worse = new_obj < old_obj - 1e-10 * (np.abs(old_obj) + 1.)
            if not worse.any():
                break
            new[worse] = .5 * (new[worse] + ph[worse])
            new_obj[worse] = objective(ya[:, worse], ma[:, worse], new[worse], m[worse])
        done = (np.abs(new - ph) < tol) | (np.abs(g1) < gtol) | ((ph <= lo) & (g1 < 0)) | ((ph >= hi) & (g1 > 0))
        phi[active] = new
        done_all[active[done]] = True
        active = active[~done]
        if not len(active):
            break
    return np.exp(-phi), done_all


def map_block(y, alpha_start, log_trend, q, offset, w, sigma2, rounds=40, tol=1e-6):
    """Joint MAP of (beta, alpha) for one gene block: the pilot's fit_map_dispersion on nb_gam's IRLS.
    Ponytail: uses nb_gam's private IRLS helpers; move into nb_gam only if shrinkage ever becomes primary."""
    with threadpool_limits(1):
        qq = nb_gam._outer(q)
        alpha = np.clip(alpha_start.copy(), nb_gam.A_MIN, nb_gam.A_MAX)
        mu = nb_gam._irls(y, q, qq, offset, w, alpha, nb_gam._constant_rate(y, offset, w))['mu']
        map_ok, stable = np.zeros(y.shape[1], bool), np.zeros(y.shape[1], bool)
        active = np.arange(y.shape[1])
        for _ in range(rounds):
            new, ok = map_alpha(y[:, active], mu[:, active], w, alpha[active], log_trend[active], sigma2)
            settled = np.abs(np.log(new) - np.log(alpha[active])) < tol
            alpha[active], map_ok[active] = new, ok
            mu[:, active] = nb_gam._irls(y[:, active], q, qq, offset, w, alpha[active], mu[:, active])['mu']
            stable[active[settled]] = True
            active = active[~settled]
            if not len(active):
                break
    return alpha, map_ok & stable


def shrunk_dispersion():
    q = nb_gam._orthobasis(DESIGNS['species']['full'])
    y = counts.tocsc()
    spans = [(a, min(a + 200, len(genes))) for a in range(0, len(genes), 200)]
    with parallel_config(backend='loky', inner_max_num_threads=1):
        parts = Parallel(n_jobs=N_JOBS)(delayed(map_block)(y[:, a:b].toarray(), alpha_ml[a:b], np.log(TREND[a:b]), q, OFFSET,
                                                           W, SIGMA2) for a, b in spans)
    return {'alpha': np.concatenate([a for a, _ in parts]), 'converged': np.concatenate([c for _, c in parts])}


SHRUNK = cached('nb_dispersion_shrunk', shrunk_dispersion, params={'trend': coefs, 'sigma2': SIGMA2}, inputs={'alpha': ALPHA_NB})
ALPHA_SHRUNK = {'alpha': SHRUNK['alpha'], 'converged': SHRUNK['converged'].astype(bool)}
NB_SHRUNK = {p: pd.DataFrame(cached(f'nb_lr_shrunk_{slug(p)}', lambda p=p: {k: v for k, v in nested_nb_lr(
    counts, {k: DESIGNS[p][k] for k in ('level', 'full')}, W, OFFSET, ALPHA_SHRUNK, {'T_spatial': ('level', 'full')},
    n_jobs=N_JOBS).items() if k != 'df'}, params={'label': p}, inputs={'alpha': ALPHA_SHRUNK}), index=genes) for p in PARTS}
MATCHED_SHRUNK = matched_a6('a6_matched_nb_shrunk', {p: NB_SHRUNK[p][['T_spatial']] for p in PARTS})
TESTS['A6 NB, shrunk α'] = a6_tests(MATCHED_SHRUNK, 'T_spatial', VIF_NB)
TESTS['A6 NB, shrunk α, Gaussian VIF'] = a6_tests(MATCHED_SHRUNK, 'T_spatial', VIF_GAUSS)
for key in ('A6 NB, shrunk α', 'A6 NB, shrunk α, Gaussian VIF'):
    CALLS[key], ROLE[key] = calls_of(TESTS[key]), 'sensitivity'
spearman_shrunk = rho(NB['species'].T_spatial, NB_SHRUNK['species'].T_spatial)
shrink_rows, SHRINK_OUTCOME = [], {}
for vif_label, unshrunk_key, shrunk_key in (('NB Pearson VIF (frozen)', 'A6 NB', 'A6 NB, shrunk α'),
                                            ('Gaussian VIF (sensitivity)', 'A6 NB, Gaussian VIF', 'A6 NB, shrunk α, Gaussian VIF')):
    pair = []
    for key, alpha_label in ((unshrunk_key, 'unshrunk ML α'), (shrunk_key, 'MAP-shrunk α')):
        row = {k: v for k, v in method_row(key).items() if k not in ('screen', 'version', 'role')}
        unshrunk = CALLS[unshrunk_key]['species']
        row = {'VIF': vif_label, 'dispersion': alpha_label, **row,
               'Jaccard with unshrunk': len(CALLS[key]['species'] & unshrunk) / len(CALLS[key]['species'] | unshrunk),
               'gene Spearman T_spatial vs unshrunk': 1. if key == unshrunk_key else spearman_shrunk,
               'failures (median-filled)': int((~(NB if key == unshrunk_key else NB_SHRUNK)['species'].converged.astype(bool)).sum())}
        pair.append(row)
    passes = all(r['relabeling rule'] == 'pass' and r['calibration'] == 'pass' for r in pair)
    SHRINK_OUTCOME[vif_label] = ('unshrunk ML α stays primary' if passes and pair[1]['Jaccard with unshrunk'] >= JACCARD_MIN
                                 else 'rule not met (a version fails a criterion): reported without choosing' if not passes
                                 else 'rule not met (Jaccard < 0.8): reported without choosing')
    for r in pair:
        r['rule outcome'] = SHRINK_OUTCOME[vif_label]
    shrink_rows += pair
SHRINK_TABLE = pd.DataFrame(shrink_rows).drop(columns=['continuous adds over steps', 'steps add over continuous'])
SHRINK_TABLE.to_csv(tables / 'dispersion_shrinkage.csv', index=False)
log_change = np.abs(np.log(SHRUNK['alpha'][use] / alpha_ml[use]))
print(f"MAP α: unsettled {int((~ALPHA_SHRUNK['converged']).sum())}; median |log(α_MAP / α_ML)| over trend genes "
      f'{np.median(log_change):.3f}; ML-at-bound genes: median MAP α {np.median(SHRUNK["alpha"][alpha_ml <= A_LO]):.4f}')
display(SHRINK_TABLE.round(4))
for unshrunk_key, shrunk_key in (('A6 NB', 'A6 NB, shrunk α'), ('A6 NB, Gaussian VIF', 'A6 NB, shrunk α, Gaussian VIF')):
    print(f'{shrunk_key}: shrunk-only', list(NAMES[sorted(CALLS[shrunk_key]['species'] - CALLS[unshrunk_key]['species'])]),
          '· unshrunk-only', list(NAMES[sorted(CALLS[unshrunk_key]['species'] - CALLS[shrunk_key]['species'])]))

## 8 · Supplementary figure

- **a. Simulation.** The share of simulated genes whose `T_spatial` reaches the real top-10% cut,
  by tertile of the species offset |log ratio|. Left: pure offsets, where every high score is a
  false shape. Right: the positive control. The dashed line marks 10%, the share of real genes
  above the cut.
- **b. Gene ranks.** Each dot is a gene: its `T_spatial` rank under the Gaussian model (x) and
  the NB model (y), as percentiles. Red: genes nearly absent in one species. Dashed lines: the
  top-10% cuts. The named genes are three proximal-tubule genes with large species offsets, and
  the three nearly absent genes whose rank falls most under NB.
- **c. Pathway calls.** The species calls of each screen under three versions: Gaussian, NB with
  the frozen NB Pearson-residual VIF, and NB with the Gaussian VIF. Black: calls the Gaussian
  version also makes. Printed on the right: calls, relabeled calls, and the share of relabeled
  p ≤ 0.05. Red: the version fails the relabeling rule or the calibration limit.
- **d. Calibration.** Observed against expected −log10 p for the 2 × 1,513 pooled relabeled
  p-values of each version. A calibrated test follows the diagonal. The legend gives the share of
  relabeled p ≤ 0.05.

In [ ]:
fig = plt.figure(figsize=(180 * MM, 150 * MM))
outer = fig.add_gridspec(2, 2, hspace=.42, wspace=.3, left=.07, right=.98, top=.95, bottom=.07)

# a · simulation
sub = outer[0, 0].subgridspec(1, 2, wspace=.12)
tertiles = ['small', 'mid', 'large']
for j, (scenario, title) in enumerate(SCENARIO_NAME.items()):
    ax = fig.add_subplot(sub[0, j])
    rows = SIM_TABLE[SIM_TABLE.scenario.eq(title)].set_index('genes')
    x = np.arange(len(tertiles) + 1)
    labels = ['all'] + tertiles
    for k, (column, color, name) in enumerate((('Gaussian share above cut', GAUSS_COLOR, 'Gaussian'),
                                               ('NB share above cut', NB_COLOR, 'NB'))):
        values = [rows.loc['all', column]] + [rows.loc[f'offset {t}', column] for t in tertiles]
        ax.bar(x + (k - .5) * .38, values, width=.36, color=color, label=name, linewidth=0)
        for xi, v in zip(x + (k - .5) * .38, values):
            ax.text(xi, v + .006, f'{v:.0%}' if v >= .005 else '0', ha='center', va='bottom', fontsize=4.8, color=INK)
    ax.axhline(TOP_SHARE, color=MUTED, lw=.6, ls='--')
    ax.set_xticks(x, labels)
    ax.set_ylim(0, max(.33, SIM_TABLE[['Gaussian share above cut', 'NB share above cut']].to_numpy().max() * 1.15))
    ax.set_title('pure offset' if scenario == 'offset' else 'positive control', loc='left', fontweight='normal')
    ax.set_xlabel('species offset tertile')
    if j == 0:
        ax.set_ylabel('share above the real top-10% cut')
        ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:.0%}'))
        ax.text(-.42, 1.08, 'a', fontsize=9, fontweight='bold', transform=ax.transAxes)
        ax.legend(loc='upper left')
    else:
        ax.set_yticklabels([])
    if j == 1:
        ax.text(3.5, TOP_SHARE + .006, '10%', ha='right', va='bottom', fontsize=5, color=MUTED)

# b · gene ranks
ax = fig.add_subplot(outer[0, 1])
rg, rn = pd.Series(gT, index=genes).rank(pct=True), pd.Series(nT, index=genes).rank(pct=True)
ax.scatter(rg[~NEARLY_ABSENT], rn[~NEARLY_ABSENT], s=1.2, color=LIGHT, linewidths=0, rasterized=True)
ax.scatter(rg[NEARLY_ABSENT], rn[NEARLY_ABSENT], s=2.2, color=ABSENT_COLOR, linewidths=0, rasterized=True, alpha=.8)
for v in (1 - TOP_SHARE,):
    ax.axvline(v, color=MUTED, lw=.6, ls='--')
    ax.axhline(v, color=MUTED, lw=.6, ls='--')
named = [g for g in ('Mep1b', 'Cyp4b1', 'Car14') if g in rg.index]
shift = (rg - rn)[NEARLY_ABSENT & TOP_G]
LABELLED = named + [g for g in shift.sort_values(ascending=False).index if g not in named][:3]
ax.scatter(rg[LABELLED], rn[LABELLED], s=14, facecolor='none', edgecolor=INK, lw=.6, zorder=3)
label_y = np.linspace(.08, .82, len(LABELLED))                     # labels outside the right edge, in rank order
for g, yt in zip(sorted(LABELLED, key=lambda g: (round(rn[g] * 20), -rg[g])), label_y):   # leftmost point -> higher label
    ax.annotate(g, xy=(rg[g], rn[g]), xytext=(1.07, yt), textcoords='axes fraction', fontsize=5.4, style='italic',
                va='center', color=ABSENT_COLOR if NEARLY_ABSENT[genes.index(g)] else INK, annotation_clip=False,
                arrowprops=dict(arrowstyle='-', lw=.4, color=MUTED, shrinkA=1, shrinkB=2))
handles = [plt.Line2D([], [], marker='o', lw=0, markersize=2.5, color=ABSENT_COLOR,
                      label=f'nearly absent in one species ({int(NEARLY_ABSENT.sum())})'),
           plt.Line2D([], [], marker='o', lw=0, markersize=2.5, color=LIGHT, label='other genes'),
           plt.Line2D([], [], marker='o', lw=0, markersize=3.5, markerfacecolor='none', markeredgecolor=INK, label='named')]
ax.legend(handles=handles, loc='upper center', bbox_to_anchor=(.5, -.13), ncol=3, handletextpad=.2, columnspacing=1)
ax.set_xlim(0, 1.005), ax.set_ylim(0, 1.005)
ax.set_xlabel('Gaussian T_spatial rank (percentile)', color=GAUSS_COLOR)
ax.set_ylabel('NB T_spatial rank (percentile)', color=NB_COLOR)
ax.set_title(f'Spearman {rho(gT, nT):.2f} · nearly absent genes in the top 10%: Gaussian '
             f'{int((NEARLY_ABSENT & TOP_G).sum())}, NB {int((NEARLY_ABSENT & TOP_N).sum())}', loc='left', fontweight='normal')
ax.text(-.16, 1.08, 'b', fontsize=9, fontweight='bold', transform=ax.transAxes)

# c · pathway calls by model
from matplotlib.colors import to_rgb


def lighter(color, amount=.5):
    r, g, b = to_rgb(color)
    return (r + (1 - r) * amount, g + (1 - g) * amount, b + (1 - b) * amount)


NB_GVIF_COLOR = lighter(NB_COLOR, .5)
VERSIONS = [('Gaussian', 'Gaussian', GAUSS_COLOR), ('NB', 'NB', NB_COLOR), ('NB, Gaussian VIF', 'NB · Gaussian VIF', NB_GVIF_COLOR)]
ax = fig.add_subplot(outer[1, 0])
ypos, ylabels, ycolors = [], [], []
y = 0
for screen in ('A6', 'A4'):
    reference_calls = CALLS[f'{screen} Gaussian']['species']
    for version, label, color in VERSIONS:
        key = f'{screen} {version}'
        species_calls = CALLS[key]['species']
        shared = len(species_calls & reference_calls) if version != 'Gaussian' else 0
        ax.barh(y, shared, color=INK, height=.62, linewidth=0)
        ax.barh(y, len(species_calls) - shared, left=shared, color=color, height=.62, linewidth=0)
        row = METHOD_TABLE.loc[key]
        verdict = 'passes' if row['relabeling rule'] == 'pass' and row['calibration'] == 'pass' else (
            'fails calibration' if row['relabeling rule'] == 'pass' else 'fails relabeling')
        ax.text(len(species_calls) + 1.5, y, f"{len(species_calls)} · relabeled {row['relabeled calls']} · "
                f"{row['calibration share']:.3f}", va='center', fontsize=5.4, color=INK)
        if verdict != 'passes':
            ax.text(len(species_calls) + 1.5, y - .36, verdict, va='center', fontsize=5, color=ABSENT_COLOR, fontweight='bold')
        ypos.append(y), ylabels.append(f'{screen}·JNT · {label}'), ycolors.append(GAUSS_COLOR if version == 'Gaussian' else NB_COLOR)
        y -= 1
    y -= .6
ax.set_yticks(ypos, ylabels)
for tick, color in zip(ax.get_yticklabels(), ycolors):
    tick.set_color(color)
ax.set_xlim(0, max(len(CALLS[f'{s} {v}']['species']) for s in ('A6', 'A4') for v, _, _ in VERSIONS) * 1.75)
ax.set_ylim(y + .9, .6)
ax.set_xlabel('species calls (BH q ≤ 0.05)')
from matplotlib.patches import Patch
gap = (ypos[2] + ypos[3]) / 2
ax.legend(handles=[Patch(color=INK, label='also called by the Gaussian version')], loc='center right',
          bbox_to_anchor=(1, (gap - (y + .9)) / (.6 - (y + .9))))
ax.set_title('calls · relabeled calls · share of relabeled p ≤ 0.05', loc='left', fontweight='normal')
ax.text(-.4, 1.08, 'c', fontsize=9, fontweight='bold', transform=ax.transAxes)

# d · calibration
sub = outer[1, 1].subgridspec(1, 2, wspace=.12)
n_pool = 2 * len(pathways)
expected = -np.log10((np.arange(1, n_pool + 1) - .5) / n_pool)
top_value = max(float((-np.log10(np.concatenate([TESTS[f'{sc} {v}'][s].p_joint.to_numpy(float) for s in SWAPS]))).max())
                for sc in ('A6', 'A4') for v, _, _ in VERSIONS)
for j, screen in enumerate(('A6', 'A4')):
    ax = fig.add_subplot(sub[0, j])
    ax.plot([0, expected[0]], [0, expected[0]], color=MUTED, lw=.7, ls='--')
    for version, label, color in VERSIONS:
        key = f'{screen} {version}'
        pooled = np.sort(np.concatenate([TESTS[key][s].p_joint.to_numpy(float) for s in SWAPS]))
        ax.plot(expected, -np.log10(pooled), color=color, lw=1.2, label=f"{label}: {METHOD_TABLE.loc[key, 'calibration share']:.3f}")
    ax.set_xlim(0, expected[0] * 1.03)
    ax.set_ylim(0, top_value * 1.05)
    ax.set_title(f'{screen}·JNT', loc='left')
    ax.set_xlabel('expected −log10 p')
    ax.legend(loc='upper center', bbox_to_anchor=(.5, -.2), title='relabeled p ≤ 0.05 (limit 0.075)', title_fontsize=5.2,
              fontsize=5.2)
    if j == 0:
        ax.set_ylabel('observed −log10 p, both relabelings')
        ax.text(-.36, 1.08, 'd', fontsize=9, fontweight='bold', transform=ax.transAxes)
    else:
        ax.set_yticklabels([])
for ext in ('png', 'pdf'):
    fig.savefig(figures / f'figS_gene_model_comparison.{ext}', bbox_inches='tight')
plt.show()

## 9 · Summary

**Gene level: the case for NB holds (sections 5 and 6).**
1. **The two models rank genes differently.** Spearman of `T_spatial` between models is 0.36,
   while `T_level` agrees (0.96).
2. **The Gaussian `T_spatial` follows the species offset.** Its Spearman with `T_level` is 0.40
   (NB 0.20), and with the size of the offset |log ratio| 0.14 (NB 0.00).
3. **The genes only the Gaussian model ranks high carry large offsets.** Of its top 10%, 527
   genes are not in the NB top 10%. Their median |log ratio| is 2.11, against 0.83 for the 527
   NB-only genes. A quarter of them (25%) are nearly absent in one species; of the NB-only genes,
   0.2% are.
4. **Nearly absent genes.** 930 genes (8.4%) are detected in < 1% of one species' structures.
   139 of them are in the Gaussian top 10% (12.5%) and 6 in the NB top 10% (0.5%).
5. **The specimen-level reference agrees with NB.** The 4,588 split-plot significant genes,
   which do not depend on the switch, are 5.1% nearly absent, below the 8.4% base rate.
6. **Simulation.** Genes given a pure constant ratio reach the Gaussian top-10% cut at 10.1%,
   and at 27% in the largest-offset tertile. Under NB the rate is 0%. With a real positional bump,
   NB scores 5.5% of genes above its cut (16.5% of highly expressed genes), and none of the pure
   offsets. The Gaussian rate with a bump (10.5%) is no higher than without it (10.1%). The
   simulation draws from the NB model, so it favours NB by construction; items 1–5 do not.

**Pathway level: the frozen NB screen fails the calibration limit on SCF13 (sections 3 and 4).**
7. **The Gaussian reference is reproduced exactly:** A6·JNT 60 calls (relabeled 2, 0; share of
   relabeled p ≤ 0.05 0.060), and A4·JNT 50 (0, 0; 0.064).
8. **NB with the Gaussian VIF reproduces the pilot:** A6·JNT 31 (0, 0; 0.067), which passes, and
   A4·JNT 15 (1, 1; 0.079), which fails calibration.
9. **NB with the frozen NB Pearson-residual VIF, computed here for the first time:**
   - A6·JNT: 74 calls, relabeled 13 and 1, share 0.085. It passes the relabeling rule
     (NCDR 0.09) but **fails the calibration limit of 0.075**.
   - A4·JNT: 62 calls, relabeled 9 and 9, share 0.105. It also fails calibration.
10. **Why the VIF changes.** The median VIF falls from 1.30 to 1.06, and the median residual
    correlation within pathways from 0.010 to 0.002. The Gaussian log1p residuals share a depth
    component: within each specimen, a structure's mean residual over all genes correlates with its
    log library at r = 0.75–0.94. The NB offset absorbs depth (r = −0.05 to −0.29). Much of the
    Gaussian residual correlation is therefore shared depth, not co-regulation. It happened to
    deflate the joint test enough to pass the relabelings. The NB residual correlation does not.
11. **Which pathways move.** 38 of the 60 Gaussian calls stay under NB (Pearson VIF). The 22
    Gaussian-only pathways have more nearly absent members (median 17% against 8%) and larger
    offsets (median |log ratio| 1.46 against 0.94–1.08).
12. **Continuous against steps.** A6 adds 13 pathways over A4 and A4 adds 3 (Gaussian). Under NB
    the numbers are 21 and 9 with the Pearson VIF, and 16 and 0 with the Gaussian VIF.

**Dispersion shrinkage (section 7).** The trend is α = 0.0239 + 0.0112 / mean, and the prior
variance is 0.666. With the Gaussian VIF the shrunk screen gives the same 31 calls (0, 0; 0.068;
Jaccard 1.0; gene Spearman 0.999), so unshrunk α stays primary, as in the pilot. With the NB
Pearson VIF both versions fail calibration (0.085 unshrunk, 0.087 shrunk), so the rule cannot be
met. Shrinkage there adds 2 calls (Jaccard 0.97).

**What this means.** The gene-level evidence supports NB counts over Gaussian log1p. But the
frozen primary pathway screen, the NB LR with the Pearson-residual VIF, fails notebook 61's
criterion 2 on SCF13. The specification's stop rule applies to notebook 37. This result should be
decided before notebook 37 is re-run. Nothing was tuned here.

**What it cannot tell you.** The relabelings mix the species, so they cannot see a species-linked
artefact. That is why the gene-level checks matter. The pilot ran on these same data before this
notebook, so the switch remains post hoc.

## 10 · Run record

In [ ]:
import platform
from importlib.metadata import version

cache_entries = []
for sidecar in sorted(cache.glob('*.key.json')):
    meta = json.loads(sidecar.read_text())
    cache_entries.append({'stage': meta['stage'], 'key': meta['key'], 'written_utc': meta['written_utc'],
                          'compute seconds': round(SECONDS.get(meta['stage'], float('nan')), 1)})
CACHE_TABLE = pd.DataFrame(cache_entries)
CACHE_TABLE = CACHE_TABLE[CACHE_TABLE.stage.isin(SECONDS)].sort_values('stage').reset_index(drop=True)
display(CACHE_TABLE)
record = {'notebook_logic_version': NOTEBOOK_LOGIC_VERSION, 'gene_model': GENE_MODEL, 'coordinate': 'SCF13',
          'n_structures': int(len(position)), 'n_genes': len(genes), 'n_pathways': len(pathways), 'n_jobs': N_JOBS,
          'live_notebook_37': LIVE37, 'guards': guard_log, 'pilot_check': pilot_check.reset_index().to_dict(orient='records'),
          'methods': METHOD_TABLE.reset_index().to_dict(orient='records'),
          'shrinkage': {'trend': coefs.tolist(), 'prior_variance': SIGMA2, 'outcome': SHRINK_OUTCOME},
          'nearly_absent': ABSENT_ROWS.reset_index().to_dict(orient='records'),
          'versions': {'python': platform.python_version(), **{m: version(m) for m in
                                                               ('numpy', 'scipy', 'pandas', 'statsmodels', 'joblib', 'matplotlib')}},
          'stage_cache': CACHE_TABLE.to_dict(orient='records'), 'compute_seconds_total': float(sum(SECONDS.values())),
          'wall_seconds': time.perf_counter() - NOTEBOOK_START}
(output / 'run_record.json').write_text(json.dumps(record, indent=2, default=str))
print(f"Wall time {record['wall_seconds']:.0f} s; summed uncached compute time of the cached stages "
      f"{record['compute_seconds_total']:.0f} s")